# Netflix Content Portfolio Analysis and Strategy Insights

**Level:** Advanced | **Tools:** pandas, matplotlib, seaborn

**Objective:** Perform a comprehensive content portfolio analysis on Netflix catalog data to understand content strategy shifts over time, country contributions, genre concentration and rating distribution.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
print('✅ Libraries imported')

## 1. Load & Clean Data

In [ ]:
df = pd.read_csv('netflix_titles.csv')
# Handle missing values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

# Parse dates
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Content Type Overview

In [ ]:
type_counts = df['type'].value_counts(normalize=True) * 100
print('Content Type Distribution:')
print(type_counts.round(1).astype(str) + '%')

## 3. Yearly Content Growth

In [ ]:
yearly_counts = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly_counts = yearly_counts.loc[2008:2021]
yearly_counts.plot(kind='bar', stacked=True, color=['#e50914', '#221f1f'], figsize=(12, 6))
plt.title('Total Titles Added per Year (Movie vs TV Show)')
plt.xlabel('Year Added')
plt.ylabel('Number of Titles')
plt.savefig('chart1_yearly_growth.png', bbox_inches='tight')
plt.show()

## 4. Strategy Shift: Movie vs TV Show %

In [ ]:
strategy_shift = yearly_counts.div(yearly_counts.sum(axis=1), axis=0) * 100
strategy_shift.plot(kind='area', stacked=True, color=['#e50914', '#221f1f'], alpha=0.8, figsize=(12, 6))
plt.title('Content Strategy Shift: % Share of Movies vs TV Shows Added')
plt.xlabel('Year Added')
plt.ylabel('Percentage (%)')
plt.legend(loc='upper right')
plt.margins(x=0)
plt.savefig('chart2_strategy_shift.png', bbox_inches='tight')
plt.show()
print('Percentage Share per Year:')
print(strategy_shift.round(1))

## 5. Genre Analysis

In [ ]:
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(15)
plt.figure(figsize=(10, 8))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='viridis')
plt.title('Top 15 Genres by Title Count')
plt.xlabel('Number of Titles')
plt.ylabel('Genre')
plt.savefig('chart3_top_genres.png', bbox_inches='tight')
plt.show()

## 6. Country Contribution

In [ ]:
df['primary_country'] = df['country'].str.split(', ').str[0]
top_countries_list = df[df['primary_country'] != 'Unknown']['primary_country'].value_counts().head(10).index
top_countries_df = df[df['primary_country'].isin(top_countries_list)]
plt.figure(figsize=(12, 6))
sns.countplot(data=top_countries_df, x='primary_country', hue='type', order=top_countries_list, palette=['#e50914', '#221f1f'])
plt.title('Top 10 Countries by Title Count (Movie vs TV Show)')
plt.xlabel('Country')
plt.ylabel('Number of Titles')
plt.xticks(rotation=45)
plt.savefig('chart4_country_contribution.png', bbox_inches='tight')
plt.show()

## 7. TV Show Season Distribution

In [ ]:
tv_shows = df[df['type'] == 'TV Show'].copy()
tv_shows['seasons'] = tv_shows['duration'].str.extract('(\d+)').astype(int)
tv_shows['season_group'] = pd.cut(tv_shows['seasons'], bins=[0, 1, 2, 3, 100], labels=['1 Season', '2 Seasons', '3 Seasons', '4+ Seasons'])
season_counts = tv_shows['season_group'].value_counts().reindex(['1 Season', '2 Seasons', '3 Seasons', '4+ Seasons'])
plt.figure(figsize=(10, 6))
sns.barplot(x=season_counts.index, y=season_counts.values, palette='mako')
plt.title('TV Show Season Distribution')
plt.xlabel('Number of Seasons')
plt.ylabel('Number of TV Shows')
plt.savefig('chart5_season_distribution.png', bbox_inches='tight')
plt.show()

## 8. Content Rating Analysis

In [ ]:
top_ratings = df['rating'].value_counts().head(10).index
plt.figure(figsize=(12, 6))
sns.countplot(data=df[df['rating'].isin(top_ratings)], x='rating', hue='type', order=top_ratings, palette=['#e50914', '#221f1f'])
plt.title('Content Rating Distribution (Movies vs TV Shows)')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.savefig('chart6_rating_distribution.png', bbox_inches='tight')
plt.show()

## 9. Monthly Additions Pattern

In [ ]:
month_counts = df['month_added'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
plt.figure(figsize=(10, 6))
sns.barplot(x=month_names, y=month_counts.values, color='#e50914')
plt.title('Content Additions by Month')
plt.xlabel('Month')
plt.ylabel('Number of Titles Added')
plt.savefig('chart7_monthly_additions.png', bbox_inches='tight')
plt.show()

## 10. Bonus: Content Age Gap

In [ ]:
df['age_gap'] = df['year_added'] - df['release_year']
age_gap_country = df[df['primary_country'].isin(top_countries_list)].groupby('primary_country')['age_gap'].mean().sort_values(ascending=False)
print('Average Content Age Gap (Years) by Top Countries:')
print(age_gap_country.round(1))

## 11. Bonus: Top Directors

In [ ]:
directors = df[df['director'] != 'Unknown']['director'].str.split(', ').explode()
top_directors = directors.value_counts().head(5)
print('Top 5 Directors by Title Count:')
print(top_directors)
print('\nGenre breakdown for top directors:')
for d in top_directors.index:
    d_genres = df[df['director'].str.contains(d, na=False, regex=False)]['listed_in'].str.split(', ').explode().value_counts().head(3)
    print(f"{d}: {', '.join([f'{k} ({v})' for k, v in d_genres.items()])}")

## 12. Key Insights

In [ ]:
print("💡 5 Netflix Content Strategy Insights:")
print("1. The catalog is movie-heavy (69.6% Movies vs 30.4% TV Shows).")
print("2. 2019 was the peak year for content additions (1543 titles).")
print("3. Netflix has steadily shifted strategy towards TV Shows: prior to 2015, Movies made up ~85% of additions, but by 2019 TV Shows grew to 32%.")
print("4. Limited series and single seasons dominate the TV space: 67% of all TV Shows have only 1 season.")
print("5. October is the most popular month for dropping new content, capitalizing on the fall TV season.")